### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from scikeras.wrappers import KerasClassifier
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import pickle

In [5]:
df = pd.read_csv('data/customer_churn_data.csv')
df.head()

df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

geography_ohe = OneHotEncoder(drop='first', sparse_output=False)
geography_ohe_data = geography_ohe.fit_transform(df[['Geography']])
geography_ohe_df = pd.DataFrame(geography_ohe_data, columns=geography_ohe.get_feature_names_out())
gender_ohe = OneHotEncoder(drop='first', sparse_output=False)
gender_ohe_data = gender_ohe.fit_transform(df[['Gender']])
gender_ohe_df = pd.DataFrame(gender_ohe_data, columns=gender_ohe.get_feature_names_out())
df.drop(columns=['Geography', 'Gender'], inplace=True)
df = pd.concat([df, gender_ohe_df, geography_ohe_df], axis=1)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [12]:
# Keras Classifier
model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)

In [13]:
# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [14]:
# Perform grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    n_jobs=-1,
    cv=3,
    verbose=1
)

In [15]:
grid_result = grid.fit(X_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: D

167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6957 - loss: 2319.5049
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6702 - loss: 1035.54483
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6520 - loss: 2111.3647
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5316 - loss: 5724.1245 
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6900 - loss: 585.2166
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6831 - loss: 48.8377
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6957 - loss: 157.0271
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6955 - loss: 228.9210
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6919 - loss: 45.1266
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7043 - loss: 196.9753
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7006 - loss: 56.1935
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 

/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 92/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6768 - loss: 33.7404Epoch 1/50
153/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6810 - loss: 32.0906Epoch 1/50


...

166/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6740 - loss: 15.7432Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6857 - loss: 18.2244
Epoch 92/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6760 - loss: 130.6581
109/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6711 - loss: 141.9614Epoch 15/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6734 - loss: 176.3138
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6745 - loss: 120.0131
Epoch 16/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6767 - loss: 176.2251
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6880 - loss: 22.6999
Epoch 93/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6807 - loss: 111.2029
Epoch 17/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6805 - loss: 137.7655
Epoch 10/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6876 - loss: 21.8056
Epoch 94/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 

/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6834 - loss: 109.4778
Epoch 24/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6724 - loss: 193.3690
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6809 - loss: 111.0548
Epoch 16/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6768 - loss: 99.2187 
Epoch 25/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6726 - loss: 153.7845
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6858 - loss: 69.0420
Epoch 26/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6827 - loss: 85.2346
Epoch 17/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6874 - loss: 81.20392
Epoch 18/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6794 - loss: 86.8377
Epoch 27/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6818 - loss: 119.9144
Epoch 10/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6790 - loss: 317.9850
Epoch 2/100
167/167 ━━━━━━━━

/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-11-15 07:25:02.615633: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 954us/step - accuracy: 0.6769 - loss: 319.9906
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - accuracy: 0.6799 - loss: 166.1250
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 945us/step - accuracy: 0.6826 - loss: 148.1495
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step - accuracy: 0.6712 - loss: 119.2008
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 973us/step - accuracy: 0.6835 - loss: 115.3773
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - accuracy: 0.6866 - loss: 128.9199
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6854 - loss: 88.7207
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 993us/step - accuracy: 0.6852 - loss: 87.4963
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step - accuracy: 0.6760 - loss: 108.5709
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 950us/step - accuracy: 0.6866 - loss: 95.4081 
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 946us/step - accuracy: 0.6848 - loss: 117.4209
Epoch 12/100


In [16]:
# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best: 0.790749 using {'epochs': 100, 'layers': 2, 'neurons': 64}
